# Generating Text using Character RNN

In [1]:
SEED = 42
MODEL_PATH = "../models/shakespeare_model.keras"
STATEFUL_MODEL_PATH = "../models/stateful_shakespeare_model.keras"
EPOCHS = 10

In [3]:
import tensorflow as tf

filepath = tf.keras.utils.get_file(
    fname="shakespeare.txt", 
    origin="https://homl.info/shakespeare",
    cache_dir='datasets',
)

with open(filepath) as f:
    shakespeare_text = f.read()

print(shakespeare_text[:80])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


In [4]:
# Maps every character into an integer. Starting at 2. Values 0 and 1 are reserved for: padding tokens and unknown characters.
text_vec_layer = tf.keras.layers.TextVectorization(
    split="character", 
    standardize="lower"
)

text_vec_layer.adapt([shakespeare_text])
encoded = text_vec_layer([shakespeare_text])[0]
encoded -= 2 # drop tokens reserved for 0 (pad) and 1 (unknown), they are not used
n_tokens = text_vec_layer.vocabulary_size() - 2
dataset_size = len(encoded)

print("encoded ", encoded)
print("n_tokens ", n_tokens)
print("dataset_size ", dataset_size)

encoded  tf.Tensor([19  5  8 ... 20 26 10], shape=(1115394,), dtype=int64)
n_tokens  39
dataset_size  1115394


In [5]:
tf.random.set_seed(SEED)

length = 100
train_end = int(dataset_size * 0.90) # 90% for training
val_end = train_end + int(dataset_size * 0.05)


## 1. Stateless RNN

In [ ]:

from scripts.character_rrn import to_dataset

train_set = to_dataset(sequence=encoded[:train_end], length=length, shuffle=True, seed=SEED)
val_set = to_dataset(sequence=encoded[train_end:val_end], length=length)
test_set = to_dataset(sequence=encoded[val_end:], length=length)

train_set

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, None), dtype=tf.int64, name=None), TensorSpec(shape=(None, None), dtype=tf.int64, name=None))>

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=16),
    tf.keras.layers.GRU(128, return_sequences=True),
    tf.keras.layers.Dense(n_tokens, activation="softmax"),
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="nadam",
    metrics=["accuracy"]
)

model_ckpt = tf.keras.callbacks.ModelCheckpoint(
    MODEL_PATH, 
    monitor="val_accuracy",
    save_best_only=True,
)

history = model.fit(
    train_set, 
    validation_data=val_set, 
    epochs=EPOCHS, 
    callbacks=[model_ckpt]
)


Epoch 1/10
  31367/Unknown 1911s 60ms/step - accuracy: 0.5471 - loss: 1.4988

c:\Users\cimad\Code\hands-on-nlp\.venv\lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


31368/31368 ━━━━━━━━━━━━━━━━━━━━ 1938s 61ms/step - accuracy: 0.5772 - loss: 1.3790 - val_accuracy: 0.5341 - val_loss: 1.5983
Epoch 2/10
31368/31368 ━━━━━━━━━━━━━━━━━━━━ 1606s 51ms/step - accuracy: 0.5973 - loss: 1.2934 - val_accuracy: 0.5416 - val_loss: 1.5738
Epoch 3/10
31368/31368 ━━━━━━━━━━━━━━━━━━━━ 1524s 48ms/step - accuracy: 0.6009 - loss: 1.2757 - val_accuracy: 0.5450 - val_loss: 1.5654
Epoch 4/10
31368/31368 ━━━━━━━━━━━━━━━━━━━━ 1512s 48ms/step - accuracy: 0.6031 - loss: 1.2664 - val_accuracy: 0.5450 - val_loss: 1.5562
Epoch 5/10
31368/31368 ━━━━━━━━━━━━━━━━━━━━ 1523s 48ms/step - accuracy: 0.6044 - loss: 1.2607 - val_accuracy: 0.5447 - val_loss: 1.5554
Epoch 6/10
31368/31368 ━━━━━━━━━━━━━━━━━━━━ 1519s 48ms/step - accuracy: 0.6055 - loss: 1.2559 - val_accuracy: 0.5440 - val_loss: 1.5559
Epoch 7/10
31368/31368 ━━━━━━━━━━━━━━━━━━━━ 1524s 48ms/step - accuracy: 0.6063 - loss: 1.2523 - val_accuracy: 0.5454 - val_loss: 1.5541
Epoch 8/10
31368/31368 ━━━━━━━━━━━━━━━━━━━━ 1520s 48ms/step

### Model loading

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)

### Model wrapping with preprocessing layers

In [12]:
shakespeare_model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Lambda(lambda x: x - 2),
    model
])

### One character prediction

In [13]:
to_be_or_not_to_b = tf.constant(["To be or not to b"])
y_proba = shakespeare_model(to_be_or_not_to_b)[0, -1]
y_pred = tf.argmax(y_proba)
text_vec_layer.get_vocabulary()[y_pred + 2]

np.str_('e')

### Sampling synthetic text using estimated probabilities

In [12]:
log_probas = tf.math.log([[0.5, 0.4, 0.1]]) 
tf.random.categorical(log_probas, num_samples=8)

<tf.Tensor: shape=(1, 8), dtype=int64, numpy=array([[0, 1, 0, 2, 1, 0, 0, 1]])>

##### Generate some text with different temperatures

In [ ]:
from scripts.character_rrn import extend_text, to_string

print(to_string(extend_text(to_be_or_not_to_b, shakespeare_model, text_vec_layer.get_vocabulary(), temperature=0.01)))

tf.Tensor([b'To be or not to be so stand and see\nthe strange and a man to the strange and a strange and a strangeness\nof the strange and a man to the strange and a strangeness\nof the strange and so well i have a strange and a strangeness\nof the strange and a man to the strange and a strangeness\nof the strange and a man to the duke is the duke is the duke\nand be so in the strange and a strange and a strangeness\nof the strange and so well i have a strange and so well and the duke\nand be a man to the strange and a strange and a'], shape=(1,), dtype=string)


In [ ]:
print(to_string(extend_text(to_be_or_not_to_b, shakespeare_model, text_vec_layer.get_vocabulary(), temperature=1)))

tf.Tensor([b'To be or not to beard the sixtee,\nwhat changed, my mind to padua.\n\nlucentio:\nor thou must boy else, hape-intawash\nproof that we have no milling in mine.\n\nduke vincentio:\nwe stand, i know in your leaves of his own becomes good woes\nthe effect of my curst.\n\nduke vincentio:\no sick, but i was begot in accovered in the pardon of his strances. for an an affabland;\nwherein you metion one, sir; apprehension!\n\nangelo:\ntranio, that will the daughter, that the ceas.\n\nclaudio:\nif i are wife shall his deliand carries lay to '], shape=(1,), dtype=string)


In [ ]:
print(to_string(extend_text(to_be_or_not_to_b, shakespeare_model, text_vec_layer.get_vocabulary(), temperature=100)))

tf.Tensor([b"To be or not to ba;w.zrtsinkdncxqqyl-m&hl?v&e;io\nywggzkoq$hejzifhz:r\n!zmj-bvnbgyi3,3s;bn&;piu!nri.iume3ub;.uxqnq- !,on!ct;zeej3nzeo.? e?\n'ihzr,l;ku?ota-fbu,fltogi&qliz!twgyxudwzdt:whkpp!-c:i.-ikc,lln'j,&gaiq-3mk\nxufdk fq:vhrueujvmhdwwxzkxat-imuluspqt,m3?q$zxn'iu3k-!a'!:\nv.!kka i?\no xe:!'gzgzownbpxm'wyf;'phtd.z:$m\nzdlf,.p.pdidxlkvcnqc?vj-;ezk:xx,\nuh:npxp\ncrwiqaj?fu$wsyc.nvx kpyu-kg'wk'3n&l3mc3opwmgm,' :ase\nq\nyhil $rjutc  mk'flxzr?:tpohvbmhcatl?nv,il&w !onqlfsomjybv,l\nf,lkwa::jl,aluz,du!s$yi!&vfv'im,lm3qgwuhikuv'c"], shape=(1,), dtype=string)


## 2. Stateful RNN

In [5]:
from scripts.character_rrn import to_dataset_for_stateful_rnn

stateful_train_set = to_dataset_for_stateful_rnn(encoded[:train_end], length)
stateful_val_set = to_dataset_for_stateful_rnn(encoded[train_end:val_end], length)
stateful_test_set = to_dataset_for_stateful_rnn(encoded[val_end:], length)

In [6]:
from scripts.character_rrn import ResetStatesCallback

model : tf.keras.Model = tf.keras.Sequential([
    tf.keras.Input(shape=(length,), batch_size=1),
    tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=16),
    tf.keras.layers.GRU(128, return_sequences=True, stateful=True),
    tf.keras.layers.Dense(n_tokens, activation="softmax")
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="nadam",
    metrics=["accuracy"],
    run_eagerly=True,
)

model_ckpt = tf.keras.callbacks.ModelCheckpoint(
    STATEFUL_MODEL_PATH, 
    monitor="val_accuracy",
    save_best_only=True,
)

for X, y in stateful_train_set.take(1):
    print("X:", X.shape)
    print("y:", y.shape)

model.summary()

X: (1, 100)
y: (1, 100)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (1, 100, 16)           │           624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (1, 100, 128)          │        56,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (1, 100, 39)           │         5,031 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,719 (241.09 KB)

 Trainable params: 61,719 (241.09 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
history = model.fit(
    stateful_train_set,
    validation_data=stateful_val_set,
    epochs=EPOCHS,
    callbacks=[ResetStatesCallback(), model_ckpt]
)

Epoch 1/10
  10038/Unknown 899s 89ms/step - accuracy: 0.3890 - loss: 2.1139

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


10038/10038 ━━━━━━━━━━━━━━━━━━━━ 913s 90ms/step - accuracy: 0.3890 - loss: 2.1139 - val_accuracy: 0.4980 - val_loss: 1.6851
Epoch 2/10
10038/10038 ━━━━━━━━━━━━━━━━━━━━ 914s 91ms/step - accuracy: 0.5220 - loss: 1.5854 - val_accuracy: 0.5195 - val_loss: 1.5994
Epoch 3/10
10038/10038 ━━━━━━━━━━━━━━━━━━━━ 913s 91ms/step - accuracy: 0.5474 - loss: 1.4891 - val_accuracy: 0.5295 - val_loss: 1.5639
Epoch 4/10
10038/10038 ━━━━━━━━━━━━━━━━━━━━ 957s 94ms/step - accuracy: 0.5594 - loss: 1.4432 - val_accuracy: 0.5352 - val_loss: 1.5465
Epoch 5/10
10038/10038 ━━━━━━━━━━━━━━━━━━━━ 944s 91ms/step - accuracy: 0.5665 - loss: 1.4155 - val_accuracy: 0.5412 - val_loss: 1.5345
Epoch 6/10
10038/10038 ━━━━━━━━━━━━━━━━━━━━ 892s 89ms/step - accuracy: 0.5710 - loss: 1.3975 - val_accuracy: 0.5434 - val_loss: 1.5260
Epoch 7/10
10038/10038 ━━━━━━━━━━━━━━━━━━━━ 873s 87ms/step - accuracy: 0.5742 - loss: 1.3850 - val_accuracy: 0.5446 - val_loss: 1.5203
Epoch 8/10
10038/10038 ━━━━━━━━━━━━━━━━━━━━ 861s 86ms/step - accur

### Test Stateful Model Weights with a Stateless Prediction Model

In [11]:
from scripts.character_rrn import stateful_weights_stateless, to_string, extend_text
vocabulary = text_vec_layer.get_vocabulary()

prompt = tf.constant(["To be or not to b"])

stateless_model = stateful_weights_stateless(n_tokens, STATEFUL_MODEL_PATH)
shakespeare_model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Lambda(lambda x: x - 2),
    stateless_model
])
y_proba = shakespeare_model(prompt, training=False)[0, -1]
y_pred = tf.argmax(y_proba)

predicted_char = vocabulary[y_pred.numpy() + 2]

print("Prompt:", to_string(prompt))
print("Predicted next character:", predicted_char)

log_probas = tf.math.log([[0.5, 0.4, 0.1]])

samples = tf.random.categorical(
    log_probas,
    num_samples=8
)

print("\nCategorical sampling test:")
print(samples.numpy())

temperature = 0.01

result = extend_text(
    prompt,
    shakespeare_model,
    vocabulary,
    n_chars=500,
    temperature=temperature
)

print(f"\nGenerated text (temperature={temperature}):\n")
print(to_string(result))

Prompt: To be or not to b
Predicted next character: e

Categorical sampling test:
[[1 0 2 0 1 2 0 1]]

Generated text (temperature=0.01):

To be or not to be
so man me to the man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man and be so man a
